# 🌟 Genesis JEPA: GPU-Accelerated Lifelong Self-Play on Crafter

This notebook runs the **Genesis JEPA** architecture on Google Colab with full GPU acceleration.

### 🛡️ Research Integrity & Anti-Cheating Invariants (AGENTS.md & GEMINI.md):
- **Zero Recipe Tables / Dictionaries**: Costs and material requirements are learned purely from empirical experience.
- **Strict Visual Boundary**: The agent decision loop receives strictly act(obs_uint8, reward_prev, step).
- **Google Drive Persistence**: Checkpoints, logs, and evaluation metrics persist in /content/drive/MyDrive/learn-a-world/genesis_jepa/.


In [ ]:
# Cell 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2: Mount Google Drive & Setup Dedicated Genesis JEPA Subfolder
from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/learn-a-world/genesis_jepa"
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
LOGS_DIR = os.path.join(DRIVE_ROOT, "logs")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Dedicated Google Drive subfolder initialized at:", DRIVE_ROOT)
print("Subdirectories ready:", os.listdir(DRIVE_ROOT))


In [ ]:
# Cell 3: Clone or Pull Repository & Install Dependencies
import os
repo_dir = "/content/learn-a-world"

if os.path.exists(repo_dir):
    %cd 
    !git pull origin main
else:
    !git clone https://github.com/sahiltolani30/learn-a-world.git 
    %cd 

!pip install --quiet crafter gymnasium pytest matplotlib


In [ ]:
# Cell 4: Run 23-Test Verification Suite & Anti-Cheating AST Integrity Audit
%cd /content/learn-a-world
!python3 -m pytest genesis_jepa/tests/ -v
!python3 genesis_jepa/audit_integrity.py --target-dirs genesis_jepa


In [ ]:
# Cell 5: Launch GPU-Accelerated Lifelong Self-Play Training
# Checkpoints and training history automatically persist to Google Drive!
%cd /content/learn-a-world

NUM_GAMES = 50          # Adjust number of self-play games (e.g. 50, 100, 200)
MAX_STEPS = 300
CKPT_PATH = f"{CHECKPOINTS_DIR}/agent_latest.pt"
HISTORY_PATH = f"{LOGS_DIR}/train_history.json"

!python3 -m genesis_jepa.train --episodes {NUM_GAMES} --max-steps {MAX_STEPS} --checkpoint "{CKPT_PATH}" --history-json "{HISTORY_PATH}"


In [ ]:
# Cell 6: Run Full Benchmark Evaluation on Crafter
%cd /content/learn-a-world
EVAL_PATH = f"{RESULTS_DIR}/eval_metrics.json"

!python3 -m genesis_jepa.evaluate --episodes 10 --max-steps 500 --seed 42 --output-json "{EVAL_PATH}"


In [ ]:
# Cell 7: Plot Training Dynamics & Milestone Learning Curves
import json
import matplotlib.pyplot as plt

if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH) as f:
        history = json.load(f)

    episodes = [h["episode"] for h in history]
    wm_loss = [h["mean_wm_loss"] for h in history]
    critic_loss = [h["mean_critic_loss"] for h in history]
    steps = [h["steps"] for h in history]
    rewards = [h["total_reward"] for h in history]

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs[0, 0].plot(episodes, wm_loss, color="#1f77b4", marker="o", lw=2)
    axs[0, 0].set_title("World Model Prediction Loss (Normalized L1)")
    axs[0, 0].set_xlabel("Game Episode")
    axs[0, 0].grid(True, alpha=0.3)

    axs[0, 1].plot(episodes, critic_loss, color="#ff7f0e", marker="s", lw=2)
    axs[0, 1].set_title("65-Bin Symlog Critic Cross-Entropy Loss")
    axs[0, 1].set_xlabel("Game Episode")
    axs[0, 1].grid(True, alpha=0.3)

    axs[1, 0].plot(episodes, steps, color="#2ca02c", marker="^", lw=2)
    axs[1, 0].axhline(220, color="red", linestyle="--", label="Nightfall Survival (220 steps)")
    axs[1, 0].set_title("Survival Steps per Episode")
    axs[1, 0].set_xlabel("Game Episode")
    axs[1, 0].legend()
    axs[1, 0].grid(True, alpha=0.3)

    axs[1, 1].plot(episodes, rewards, color="#d62728", marker="D", lw=2)
    axs[1, 1].set_title("Cumulative Episode Reward")
    axs[1, 1].set_xlabel("Game Episode")
    axs[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = f"{RESULTS_DIR}/genesis_learning_curves.png"
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print(f"Plots saved to {plot_path}")
else:
    print("No history file found yet. Run Cell 5 first!")
